**IMPORTATION DES LIBRAIRIES**

## 📌 Résumé du projet  ##

Ce notebook présente l’implémentation d’un réseau de neurones entièrement codé à la main en NumPy, entraîné sur des données simulées (make_blobs). L'objectif est de comprendre les mécanismes de base : propagation, rétropropagation, descente de gradient et visualisation de la frontière de décision.

- Bibliothèques utilisées : NumPy, Matplotlib, Scikit-learn
- Visualisation dynamique de la frontière de décision
- Accuracy (Précision) obtenue : ~93%


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from IPython.display import display, clear_output
import time

**FONCTION D'INITIALISATION**

In [ ]:
def initialisation(n0, n1, n2):
    W1 = np.random.randn(n1, n0) * 0.01
    b1 = np.random.randn(n1, 1) * 0.01
    W2 = np.random.randn(n2, n1) * 0.01
    b2 = np.random.randn(n2, 1) * 0.01

    parametres = {
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2
    }
    return parametres

**FONCTION DE FORWARD PROPAGATION**

In [ ]:
def forward_propagation(X, parametres):
    W1 = parametres["W1"]
    b1 = parametres["b1"]
    W2 = parametres["W2"]
    b2 = parametres["b2"]

    Z1 = W1.dot(X) + b1
    A1 = 1 / (1 + np.exp(-Z1))
    Z2 = W2.dot(A1) + b2
    A2 = 1 / (1 + np.exp(-Z2))

    activations = {
        "A1": A1,
        "A2": A2
    }
    return activations

**FONCTION DE BACKWARD PROPAGATION**

In [ ]:
def backward_propagation(X, y, parametres, activations):
    A1 = activations["A1"]
    A2 = activations["A2"]
    W2 = parametres["W2"]

    m = y.shape[1]
    dZ2 = A2 - y
    dW2 = 1/m * dZ2.dot(A1.T)
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)
    # Corrected the derivative of the sigmoid function
    dZ1 = np.dot(W2.T, dZ2) * (A1 * (1 - A1))
    dW1 = 1/m * dZ1.dot(X.T)
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {
        "dW1": dW1,
        "dW2": dW2,
        "db1": db1,
        "db2": db2
    }
    return gradients

**FONCTION D'UPDATE**

In [ ]:
def update(gradients, parametres, learning_rate):
    # Used element-wise subtraction for parameters update
    parametres["W1"] -= learning_rate * gradients["dW1"]
    parametres["b1"] -= learning_rate * gradients["db1"]
    parametres["W2"] -= learning_rate * gradients["dW2"]
    parametres["b2"] -= learning_rate * gradients["db2"]

    return parametres

**LE LOG LOSS**

In [ ]:
def log_loss(A, y):
    # Added check for empty input to prevent errors
    if y.shape[1] == 0:
        return 0.0
    # Added a small epsilon to prevent division by zero or log of zero
    epsilon = 1e-15
    return -np.mean(y * np.log(A + epsilon) + (1 - y) * np.log(1 - A + epsilon))

**FONCTION DE PREDICTION**

In [ ]:
def predict(X, parametres):
    activations = forward_propagation(X, parametres)
    A2 = activations["A2"]
    # Ensured the output is of integer type
    return (A2 >= 0.5).astype(int)

**INITALISATON DU RESEAU DE NEURONNE**

In [ ]:
def neural_network(X_train, y_train, learning_rate=0.1, n_iter=100, n1=16):

    n0 = X_train.shape[0]
    n2 = y_train.shape[0]
    parametres = initialisation(n0, n1, n2)

    W1=parametres["W1"]
    W2=parametres["W2"]
    b1=parametres["b1"]
    b2=parametres["b2"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    plt.ion()
    fig.suptitle('Évolution de l\'entraînement')

    line_loss, = ax1.plot([], [], 'r-', label='Loss')
    line_acc, = ax2.plot([], [], 'b-', label='Accuracy')

    for ax in (ax1, ax2):
        ax.set_xlabel('Itérations (x10)')
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend()

    ax1.set_ylabel('Loss')
    ax2.set_ylabel('Accuracy')

    train_loss = []
    train_acc = []

    for i in range(n_iter):
        # Forward et backward propagation
        activations = forward_propagation(X_train, parametres)
        gradients = backward_propagation(X_train, y_train, parametres, activations)
        parametres = update(gradients, parametres, learning_rate)

        if i % 10 == 0:
            loss = log_loss(activations["A2"], y_train)
            y_pred = predict(X_train, parametres)
            acc = accuracy_score(y_train.flatten(), y_pred.flatten())

            train_loss.append(loss)
            train_acc.append(acc)

            line_loss.set_data(range(len(train_loss)), train_loss)
            line_acc.set_data(range(len(train_acc)), train_acc)

            ax1.relim()
            ax1.autoscale_view()
            ax2.set_ylim(0, 1.05)
            ax2.set_xlim(0, len(train_acc))

            fig.canvas.draw()
            fig.canvas.flush_events()
            time.sleep(0.01)
    print("\033[32m Précision:",acc,"\033[0m")

    plt.show()


    x_min, x_max = X_train[0, :].min() - 1, X_train[0, :].max() + 1
    y_min, y_max = X_train[1, :].min() - 1, X_train[1, :].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))


    grid_points = np.c_[xx.ravel(), yy.ravel()].T
    activations = forward_propagation(grid_points, parametres)
    Z = activations["A2"]
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, levels=[0, 0.5, 1], cmap=plt.cm.Spectral, alpha=0.4)
    plt.scatter(X[0, :], X[1, :], c=y[0], cmap=plt.cm.Spectral, edgecolors='k')
    plt.title("Frontière de décision du réseau de neurones")
    plt.xlabel("x₁")
    plt.ylabel("x₂")
    plt.show()

    return parametres

**EXECUTION ET RESULTATS**

In [ ]:
X, y = make_blobs(n_samples=100, n_features=2, centers=2, random_state=0)
X = X.T
y = y.reshape((1, y.shape[0]))
parametres = neural_network(X, y)